In [2]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.feature_engineering import prepare_features

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

train = pd.read_parquet(
    PROCESSED_DIR / "train_canonical.parquet"
)

TARGET_COLUMN = "Цена"

X_eda = prepare_features(
    train.drop(columns=[TARGET_COLUMN])
)

eda = pd.concat(
    [
        X_eda,
        train[[TARGET_COLUMN]],
    ],
    axis=1,
)

eda.shape

(8340, 30)

In [3]:
eda[
    [
        "Год выпуска",
        "Пробег_число",
        "Двигатель_объём_л",
        "Двигатель_цилиндры",
        "Расход_л_на_100км",
        "Цена",
    ]
].describe(percentiles=[0.01, 0.05, 0.95, 0.99])

,Год выпуска,Пробег_число,Двигатель_объём_л,Двигатель_цилиндры,Расход_л_на_100км,Цена
count,8340.000000,8066.0,7539.0,7478.0,7528.0,8.340000e+03
mean,2016.247122,100082.195884,2.399668,4.445841,7.651355,3.712697e+04
std,5.152358,78269.782574,0.920179,1.049247,2.301246,3.679723e+04
min,1959.000000,1.0,0.0,2.0,0.0,9.000000e+02
1%,2001.000000,9.65,1.0,3.0,0.0,5.995000e+03
5%,2007.000000,45.25,1.3,4.0,4.6,9.499000e+03
95%,2023.000000,245205.75,4.0,6.0,11.5,8.688800e+04
99%,2023.000000,329065.45,6.0,8.0,14.4,1.694980e+05
max,2023.000000,526162.0,9.8,12.0,20.7,1.500000e+06


сначала проверяем, отличаются ли машины с пропущенным пробегом по цене и году выпуска.

In [4]:
missing_feature_report = []

features_to_check = [
    "Пробег_число",
    "Двигатель_объём_л",
    "Двигатель_цилиндры",
    "Расход_л_на_100км",
]

for feature in features_to_check:
    is_missing = eda[feature].isna()

    missing_feature_report.append(
        {
            "feature": feature,
            "missing_count": int(is_missing.sum()),
            "missing_share_pct": round(is_missing.mean() * 100, 2),
            "median_price_missing": round(
                eda.loc[is_missing, "Цена"].median(),
                2,
            ),
            "median_price_not_missing": round(
                eda.loc[~is_missing, "Цена"].median(),
                2,
            ),
            "median_year_missing": round(
                eda.loc[is_missing, "Год выпуска"].median(),
                1,
            ),
            "median_year_not_missing": round(
                eda.loc[~is_missing, "Год выпуска"].median(),
                1,
            ),
        }
    )

missing_feature_report = pd.DataFrame(missing_feature_report)
display(missing_feature_report)

,feature,missing_count,missing_share_pct,median_price_missing,median_price_not_missing,median_year_missing,median_year_not_missing
0,Пробег_число,274,3.29,60826.5,28990.0,2023.0,2017.0
1,Двигатель_объём_л,801,9.60,29830.0,29500.0,2015.0,2017.0
2,Двигатель_цилиндры,862,10.34,29999.0,28999.0,2016.0,2017.0
3,Расход_л_на_100км,812,9.74,28999.0,29690.0,2015.0,2017.0


In [5]:
parser_audit = pd.DataFrame(
    {
        "metric": [
            "Двигатель указан, но объём не извлечён",
            "Двигатель указан, но цилиндры не извлечены",
            "Расход указан, но число не извлечено",
        ],
        "count": [
            int(
                eda["Двигатель"].notna().sum()
                - (
                    eda["Двигатель"].notna()
                    & eda["Двигатель_объём_л"].notna()
                ).sum()
            ),
            int(
                eda["Двигатель"].notna().sum()
                - (
                    eda["Двигатель"].notna()
                    & eda["Двигатель_цилиндры"].notna()
                ).sum()
            ),
            int(
                eda["Расход"].notna().sum()
                - (
                    eda["Расход"].notna()
                    & eda["Расход_л_на_100км"].notna()
                ).sum()
            ),
        ],
    }
)

display(parser_audit)

,metric,count
0,"Двигатель указан, но объём не извлечён",0
1,"Двигатель указан, но цилиндры не извлечены",61
2,"Расход указан, но число не извлечено",0


In [6]:
display(
    eda.loc[
        eda["Двигатель"].notna()
        & eda["Двигатель_цилиндры"].isna(),
        [
            "Двигатель",
            "Топливо",
            "Полное название",
            "Бренд",
            "Модель",
            "Цена",
        ],
    ].head(30)
)

,Двигатель,Топливо,Полное название,Бренд,Модель,Цена
231,0 L,ELECTRIC,2022 VOLVO C40 RECHARGE PURE ELECTRIC,VOLVO,C40,79990
254,0 L,ELECTRIC,2023 KIA EV6 GT-LINE AWD (WITH SUNROOF),KIA,EV6,87590
257,0 L,ELECTRIC,2023 AUDI E-TRON RS GT QUATTRO,AUDI,E-TRON,286688
913,0 L,ELECTRIC,2023 PORSCHE TAYCAN 4S CROSS TURISMO,PORSCHE,TAYCAN,209600
1837,0 L,ELECTRIC,2022 AUDI E-TRON RS GT QUATTRO,AUDI,E-TRON,266990
2040,0 L,ELECTRIC,2022 BMW IX XDRIVE40 SPORT,BMW,IX,129888
2155,0 L,ELECTRIC,2021 KIA NIRO ELECTRIC SPORT,KIA,NIRO,46990
2272,0 L,ELECTRIC,2023 KIA EV6 AIR RWD,KIA,EV6,72590
2464,0 L,ELECTRIC,2022 AUDI E-TRON 55 QUATTRO,AUDI,E-TRON,159990
2471,0 L,ELECTRIC,2022 PORSCHE TAYCAN TURBO S,PORSCHE,TAYCAN,339900


подтвердим, что нули действительно почти полностью относятся к EV.

In [7]:
zero_tech_report = (
    eda.assign(
        Нулевой_объём=eda["Двигатель_объём_л"].eq(0),
        Нулевой_расход=eda["Расход_л_на_100км"].eq(0),
    )
    .groupby("Топливо", dropna=False)
    .agg(
        cars=("Цена", "size"),
        zero_engine_volume=("Нулевой_объём", "sum"),
        zero_consumption=("Нулевой_расход", "sum"),
        median_price=("Цена", "median"),
    )
    .sort_values("cars", ascending=False)
)

display(zero_tech_report)

,cars,zero_engine_volume,zero_consumption,median_price
Топливо,,,,
UNLEADED,3463,0,25,23490.0
DIESEL,2466,5,93,34999.0
PREMIUM,1665,0,8,35990.0
HYBRID,334,0,0,46638.0
<NA>,319,1,0,33690.0
ELECTRIC,66,60,60,58675.0
OTHER,17,0,0,29999.0
LPG,9,0,0,15990.0
LEADED,1,0,0,39990.0


In [8]:
display(
    eda.loc[
        (eda["Двигатель_объём_л"] == 0)
        | (eda["Расход_л_на_100км"] == 0),
        [
            "Топливо",
            "Двигатель",
            "Расход",
            "Двигатель_объём_л",
            "Двигатель_цилиндры",
            "Расход_л_на_100км",
            "Цена",
        ],
    ].head(30)
)

,Топливо,Двигатель,Расход,Двигатель_объём_л,Двигатель_цилиндры,Расход_л_на_100км,Цена
151,UNLEADED,"8 cyl, 6.2 L",0 L / 100 km,6.2,8,0.0,115990
166,PREMIUM,"6 cyl, 2.5 L",0 L / 100 km,2.5,6,0.0,27250
169,UNLEADED,"8 cyl, 6.2 L",0 L / 100 km,6.2,8,0.0,129990
180,DIESEL,"6 cyl, 7.8 L",0 L / 100 km,7.8,6,0.0,131088
231,ELECTRIC,0 L,0 L / 100 km,0.0,<NA>,0.0,79990
254,ELECTRIC,0 L,0 L / 100 km,0.0,<NA>,0.0,87590
257,ELECTRIC,0 L,0 L / 100 km,0.0,<NA>,0.0,286688
450,DIESEL,"6 cyl, 9.8 L",0 L / 100 km,9.8,6,0.0,293131
491,UNLEADED,"8 cyl, 5.7 L",0 L / 100 km,5.7,8,0.0,19990
492,UNLEADED,"4 cyl, 2.7 L",0 L / 100 km,2.7,4,0.0,19913


In [9]:
from src.feature_engineering import add_age_mileage_features

age_mileage_eda = add_age_mileage_features(X_eda.copy())
age_mileage_eda["Цена"] = train["Цена"]

посмотрю монотонность связи:

In [10]:
age_mileage_eda = add_age_mileage_features(X_eda.copy())
age_mileage_eda["Цена"] = train["Цена"]

age_mileage_eda[
    [
        "Цена",
        "Год выпуска",
        "Возраст_авто",
        "Пробег_число",
        "Лог_пробег",
        "Пробег_на_год",
        "Лог_пробег_на_год",
    ]
].corr(method="spearman")["Цена"].sort_values()

Возраст_авто        -0.715022
Пробег_число        -0.637056
Лог_пробег          -0.637056
Пробег_на_год       -0.290358
Лог_пробег_на_год   -0.290358
Год выпуска          0.715022
Цена                 1.000000
Name: Цена, dtype: float64

In [11]:
age_mileage_eda = add_age_mileage_features(X_eda.copy())
age_mileage_eda["Цена"] = train["Цена"]

age_mileage_eda[
    [
        "Цена",
        "Год выпуска",
        "Возраст_авто",
        "Пробег_число",
        "Лог_пробег",
        "Пробег_на_год",
        "Лог_пробег_на_год",
    ]
].corr(method="pearson")["Цена"].sort_values()

Пробег_число        -0.390935
Возраст_авто        -0.335712
Лог_пробег          -0.324220
Лог_пробег_на_год   -0.251576
Пробег_на_год       -0.237417
Год выпуска          0.335712
Цена                 1.000000
Name: Цена, dtype: float64

сгруппирую автомобили по возрасту:

In [12]:
age_bins = [0, 1, 3, 5, 8, 12, 20, 30, 100]

age_mileage_eda["Возрастная_группа"] = pd.cut(
    age_mileage_eda["Возраст_авто"],
    bins=age_bins,
    include_lowest=True,
)

age_report = (
    age_mileage_eda
    .groupby("Возрастная_группа", observed=False)
    .agg(
        cars=("Цена", "size"),
        median_price=("Цена", "median"),
        median_mileage=("Пробег_число", "median"),
        median_mileage_per_year=("Пробег_на_год", "median"),
    )
    .round(2)
)

display(age_report)

,cars,median_price,median_mileage,median_mileage_per_year
Возрастная_группа,,,,
"(-0.001, 1.0]",614,56460.0,20.0,20.0
"(1.0, 3.0]",1218,46999.0,14100.0,5632.0
"(3.0, 5.0]",1350,36880.0,56321.0,12486.88
"(5.0, 8.0]",2034,29999.0,87881.0,12838.58
"(8.0, 12.0]",1745,19990.0,131876.0,12872.33
"(12.0, 20.0]",1210,12990.0,179868.0,11965.0
"(20.0, 30.0]",162,12494.5,205826.5,8676.62
"(30.0, 100.0]",7,39990.0,117552.0,3358.63


In [13]:
train.columns

Index(['car_id', 'Бренд', 'Год выпуска', 'Модель', 'Тип машины',
       'Полное название', 'Исползование', 'КПП', 'Двигатель', 'Привод',
       'Топливо', 'Расход', 'Пробег', 'Цвет', 'Локация',
       'Количество цилиндров', 'Тип кузова', 'Двери', 'Количество кресел',
       'Оценка эксперта', 'Количество владельцев', 'Предложение', 'Цена'],
      dtype='str')

In [15]:
pd.set_option("display.max_columns", None)
train.head(5)

,car_id,Бренд,Год выпуска,Модель,Тип машины,Полное название,Исползование,КПП,Двигатель,Привод,Топливо,Расход,Пробег,Цвет,Локация,Количество цилиндров,Тип кузова,Двери,Количество кресел,Оценка эксперта,Количество владельцев,Предложение,Цена
0,65e4207d-80c1-47a9-9ce8-51a5e5cfca5c,GWM,2022.0,HAVAL,SPRINGWOOD GWM HAVAL,2022 GWM HAVAL H6 ULTRA AWD,DEMO,Automatic,"4 cyl, 2 L",AWD,PREMIUM,9.8 L / 100 km,244,Grey / Black,"SPRINGWOOD, QLD",4 cyl,SUV,4 Doors,5 Seats,4.0,5.0,32145.0,38812
1,331006ba-098d-4b5b-9f05-3cf9544c30ec,FORD,2014.0,TERRITORY,SUV,2014 FORD TERRITORY TITANIUM (4X4),USED,Automatic,"6 cyl, 2.7 L",AWD,DIESEL,9 L / 100 km,202408,White / -,"PENRITH, NSW",6 cyl,SUV,4 Doors,7 Seats,6.0,6.0,15949.0,15950
2,43a44ee6-5293-4a42-a04f-faf3bf0967b4,NISSAN,2021.0,X-TRAIL,SUV,2021 NISSAN X-TRAIL TI (4WD),USED,Automatic,"4 cyl, 2.5 L",4WD,UNLEADED,8.3 L / 100 km,23301,White / -,"WEST FOOTSCRAY, VIC",4 cyl,SUV,4 Doors,5 Seats,4.0,7.0,39616.0,41990
3,383affa3-d8da-4ad0-9bd7-91f48c1c00ac,VOLVO,2022.0,XC60,SUV,2022 VOLVO XC60 B5 MOMENTUM MHEV,USED,Automatic,"4 cyl, 2 L",AWD,HYBRID,7.6 L / 100 km,17359,White / Black,"SOUTH BUNBURY, WA",4 cyl,SUV,4 Doors,5 Seats,1.0,9.0,70449.0,69900
4,e100c33a-b78c-479d-8edf-5734b2bde7fd,RENAULT,2018.0,KOLEOS,SUV,2018 RENAULT KOLEOS INTENS (4X4),USED,Automatic,"4 cyl, 2 L",4WD,DIESEL,6.1 L / 100 km,100456,White / -,"RINGWOOD, VIC",4 cyl,SUV,4 Doors,5 Seats,1.0,8.0,22895.0,27950


In [17]:

from src.feature_engineering import (
    prepare_features,
    add_title_hierarchy_features,
)

prepare_features(train) 
add_title_hierarchy_features(train)

,car_id,Бренд,Год выпуска,Модель,Тип машины,Полное название,Исползование,КПП,Двигатель,Привод,Топливо,Расход,Пробег,Цвет,Локация,Количество цилиндров,Тип кузова,Двери,Количество кресел,Оценка эксперта,Количество владельцев,Предложение,Цена,Название_без_года,Название_число_слов,Название_есть_4X4,Название_есть_AWD,Название_есть_TURBO,Название_есть_SPORT,Название_есть_HYBRID,Название_есть_GT,Название_есть_LUXURY,Название_есть_DIESEL,Название_нормализованное_без_года,Название_префикс_2,Название_префикс_3,Название_префикс_4,Название_мощность_kw,Название_есть_мощность_kw,Название_моторный_маркер,Название_есть_AMG,Название_есть_M_SPORT,Название_есть_RS,Название_есть_S_LINE,Название_есть_GTI,Название_есть_HSE,Название_есть_SR5,Название_есть_GXL,Название_есть_LIMITED,Название_есть_PREMIUM,Название_есть_COMFORTLINE,Название_есть_ASCENT,Название_есть_ACTIVE,Название_есть_ELITE,Название_есть_TDI,Название_есть_TSI,Название_есть_TFSI,Название_есть_CDI,Название_есть_V6,Название_есть_V8
0,65e4207d-80c1-47a9-9ce8-51a5e5cfca5c,GWM,2022.0,HAVAL,SPRINGWOOD GWM HAVAL,2022 GWM HAVAL H6 ULTRA AWD,DEMO,Automatic,"4 cyl, 2 L",AWD,PREMIUM,9.8 L / 100 km,244,Grey / Black,"SPRINGWOOD, QLD",4 cyl,SUV,4 Doors,5 Seats,4.0,5.0,32145.0,38812,GWM HAVAL H6 ULTRA AWD,5,0,1,0,0,0,0,0,0,GWM HAVAL H6 ULTRA AWD,GWM HAVAL,GWM HAVAL H6,GWM HAVAL H6 ULTRA,NaN,0,<NA>,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,331006ba-098d-4b5b-9f05-3cf9544c30ec,FORD,2014.0,TERRITORY,SUV,2014 FORD TERRITORY TITANIUM (4X4),USED,Automatic,"6 cyl, 2.7 L",AWD,DIESEL,9 L / 100 km,202408,White / -,"PENRITH, NSW",6 cyl,SUV,4 Doors,7 Seats,6.0,6.0,15949.0,15950,FORD TERRITORY TITANIUM (4X4),4,1,0,0,0,0,0,0,0,FORD TERRITORY TITANIUM 4X4,FORD TERRITORY,FORD TERRITORY TITANIUM,FORD TERRITORY TITANIUM 4X4,NaN,0,<NA>,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,43a44ee6-5293-4a42-a04f-faf3bf0967b4,NISSAN,2021.0,X-TRAIL,SUV,2021 NISSAN X-TRAIL TI (4WD),USED,Automatic,"4 cyl, 2.5 L",4WD,UNLEADED,8.3 L / 100 km,23301,White / -,"WEST FOOTSCRAY, VIC",4 cyl,SUV,4 Doors,5 Seats,4.0,7.0,39616.0,41990,NISSAN X-TRAIL TI (4WD),5,0,0,0,0,0,0,0,0,NISSAN X TRAIL TI 4WD,NISSAN X,NISSAN X TRAIL,NISSAN X TRAIL TI,NaN,0,<NA>,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,383affa3-d8da-4ad0-9bd7-91f48c1c00ac,VOLVO,2022.0,XC60,SUV,2022 VOLVO XC60 B5 MOMENTUM MHEV,USED,Automatic,"4 cyl, 2 L",AWD,HYBRID,7.6 L / 100 km,17359,White / Black,"SOUTH BUNBURY, WA",4 cyl,SUV,4 Doors,5 Seats,1.0,9.0,70449.0,69900,VOLVO XC60 B5 MOMENTUM MHEV,5,0,0,0,0,0,0,0,0,VOLVO XC60 B5 MOMENTUM MHEV,VOLVO XC60,VOLVO XC60 B5,VOLVO XC60 B5 MOMENTUM,NaN,0,<NA>,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,e100c33a-b78c-479d-8edf-5734b2bde7fd,RENAULT,2018.0,KOLEOS,SUV,2018 RENAULT KOLEOS INTENS (4X4),USED,Automatic,"4 cyl, 2 L",4WD,DIESEL,6.1 L / 100 km,100456,White / -,"RINGWOOD, VIC",4 cyl,SUV,4 Doors,5 Seats,1.0,8.0,22895.0,27950,RENAULT KOLEOS INTENS (4X4),4,1,0,0,0,0,0,0,0,RENAULT KOLEOS INTENS 4X4,RENAULT KOLEOS,RENAULT KOLEOS INTENS,RENAULT KOLEOS INTENS 4X4,NaN,0,<NA>,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8335,6d5e13e6-33dc-4c92-866b-a994954597d0,JAGUAR,2023.0,F-TYPE,NEW AVAILABLE TO ORDER,2023 JAGUAR F-TYPE 75 P450 RWD (331KW),NEW,Automatic,"8 cyl, 5 L",Rear,PREMIUM,10.6 L / 100 km,- / -,5 years / Unlimited km,NaN,8 cyl,CONVERTIBLE,2 Doors,2 Seats,3.0,3.0,176874.0,188450,JAGUAR F-TYPE 75 P450 RWD (331KW),7,0,0,0,0,0,0,0,0,JAGUAR F TYPE 75 P450 RWD 331KW,JAGUAR F,JAGUAR F TYPE,JAGUAR F TYPE 75,331.0,1,<NA>,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
8336,78af6521-cf0a-441d-a7d0-2257c4dec010,HYUNDAI,2010.0,I30,HATCHBACK,2010 HYUNDAI I30 SX,USED,Automatic,"4 cyl, 2 L",Front,UNLEADED,7.6 L / 100 km,186939,White / Grey,"MINCHINBURY, NSW",4 cyl,HATCHBACK,5 Doors,5 Seats,7.0,5.0,8876.0,10995,HYUNDAI I30 SX,3,0,0,0,0,0,0,

In [18]:
prepare_features(train) 


,car_id,Бренд,Год выпуска,Модель,Тип машины,Полное название,Исползование,КПП,Двигатель,Привод,Топливо,Расход,Пробег,Цвет,Локация,Количество цилиндров,Тип кузова,Двери,Количество кресел,Оценка эксперта,Количество владельцев,Предложение,Цена,Пробег_число,Расход_л_на_100км,Двигатель_цилиндры,Двигатель_объём_л,Двери_число,Кресла_число,Штат
0,65e4207d-80c1-47a9-9ce8-51a5e5cfca5c,GWM,2022.0,HAVAL,SPRINGWOOD GWM HAVAL,2022 GWM HAVAL H6 ULTRA AWD,DEMO,Automatic,"4 cyl, 2 L",AWD,PREMIUM,9.8 L / 100 km,244,Grey / Black,"SPRINGWOOD, QLD",4 cyl,SUV,4 Doors,5 Seats,4.0,5.0,32145.0,38812,244,9.8,4,2.0,4,5,QLD
1,331006ba-098d-4b5b-9f05-3cf9544c30ec,FORD,2014.0,TERRITORY,SUV,2014 FORD TERRITORY TITANIUM (4X4),USED,Automatic,"6 cyl, 2.7 L",AWD,DIESEL,9 L / 100 km,202408,White / -,"PENRITH, NSW",6 cyl,SUV,4 Doors,7 Seats,6.0,6.0,15949.0,15950,202408,9.0,6,2.7,4,7,NSW
2,43a44ee6-5293-4a42-a04f-faf3bf0967b4,NISSAN,2021.0,X-TRAIL,SUV,2021 NISSAN X-TRAIL TI (4WD),USED,Automatic,"4 cyl, 2.5 L",4WD,UNLEADED,8.3 L / 100 km,23301,White / -,"WEST FOOTSCRAY, VIC",4 cyl,SUV,4 Doors,5 Seats,4.0,7.0,39616.0,41990,23301,8.3,4,2.5,4,5,VIC
3,383affa3-d8da-4ad0-9bd7-91f48c1c00ac,VOLVO,2022.0,XC60,SUV,2022 VOLVO XC60 B5 MOMENTUM MHEV,USED,Automatic,"4 cyl, 2 L",AWD,HYBRID,7.6 L / 100 km,17359,White / Black,"SOUTH BUNBURY, WA",4 cyl,SUV,4 Doors,5 Seats,1.0,9.0,70449.0,69900,17359,7.6,4,2.0,4,5,WA
4,e100c33a-b78c-479d-8edf-5734b2bde7fd,RENAULT,2018.0,KOLEOS,SUV,2018 RENAULT KOLEOS INTENS (4X4),USED,Automatic,"4 cyl, 2 L",4WD,DIESEL,6.1 L / 100 km,100456,White / -,"RINGWOOD, VIC",4 cyl,SUV,4 Doors,5 Seats,1.0,8.0,22895.0,27950,100456,6.1,4,2.0,4,5,VIC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8335,6d5e13e6-33dc-4c92-866b-a994954597d0,JAGUAR,2023.0,F-TYPE,NEW AVAILABLE TO ORDER,2023 JAGUAR F-TYPE 75 P450 RWD (331KW),NEW,Automatic,"8 cyl, 5 L",Rear,PREMIUM,10.6 L / 100 km,- / -,5 years / Unlimited km,<NA>,8 cyl,CONVERTIBLE,2 Doors,2 Seats,3.0,3.0,176874.0,188450,<NA>,10.6,8,5.0,2,2,<NA>
8336,78af6521-cf0a-441d-a7d0-2257c4dec010,HYUNDAI,2010.0,I30,HATCHBACK,2010 HYUNDAI I30 SX,USED,Automatic,"4 cyl, 2 L",Front,UNLEADED,7.6 L / 100 km,186939,White / Grey,"MINCHINBURY, NSW",4 cyl,HATCHBACK,5 Doors,5 Seats,7.0,5.0,8876.0,10995,186939,7.6,4,2.0,5,5,NSW
8337,ade7a087-c620-49f2-a69e-f1238e74f453,NISSAN,2021.0,370Z,COUPE,2021 NISSAN 370Z (5YR),USED,Automatic,"6 cyl, 3.7 L",Rear,PREMIUM,10.4 L / 100 km,7775,Black / Black,"SOUTH GEELONG, VIC",6 cyl,COUPE,2 Doors,2 Seats,7.0,2.0,58897.0,59990,7775,10.4,6,3.7,2,2,VIC
8338,302d0371-067b-4766-9ff1-71e6626ea6cf,FORD,2017.0,RANGER,UTE / TRAY,2017 FORD RANGER XLS 3.2 (4X4),USED,Automatic,"5 cyl, 3.2 L",4WD,DIESEL,9.2 L / 100 km,143738,Black / Ebony Circuit Fabric,"TWEED HEADS SOUTH, NSW",5 cyl,UTE / TRAY,4 Doors,5 Seats,9.0,1.0,36698.0,42990,143738,9.2,5,3.2,4,5,NSW
